# テーマ4 分類の基礎

## 目的
- 分類が「入力データをカテゴリに割り当てる問題」であることを理解する
- Breast Cancer データセットで、良性・悪性を分類する流れを体験する
- Accuracy と混同行列を使って、正解率だけでは見えない誤分類の中身を確認する

## 今日の成果物
- 回帰と分類の違いを1文で説明する
- 分類モデルの Accuracy を確認する
- 混同行列から、どのクラスをどう間違えたかを説明する


## 分類の基本

分類は、入力データをあらかじめ決められたカテゴリに分ける問題です。回帰が数値を予測するのに対して、分類ではクラスを予測します。

例:

- メールがスパムかどうか
- 画像に写っている数字が何か
- 腫瘍データが良性か悪性か

この notebook では、医療判断を行うのではなく、分類モデルの評価方法を学ぶために Breast Cancer データセットを使います。


### Step 0: ライブラリを読み込む
分類で使うライブラリを準備します。ロジスティック回帰、k近傍法、決定木を同じ条件で比較します。


In [ ]:
# ===== Step 0: ライブラリ読み込み =====
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

sns.set_theme(style='whitegrid')


### Step 1: データを読み込む
Breast Cancer データセットを読み込み、分類の対象となるクラス名を確認します。この授業では診断ではなく、分類モデルの評価を学ぶ教材として扱います。


In [ ]:
# ===== Step 1: データ読み込み =====
cancer = load_breast_cancer(as_frame=True)
df = cancer.frame.copy()
target_column = 'target'

print('クラス名 =', list(cancer.target_names))
print('行数, 列数 =', df.shape)
display(df.head())


### Step 2: 学習データを準備する
説明変数と目的変数を分けて、学習用とテスト用に分割します。分類ではクラスの偏りを保つため、`stratify` も確認します。


#### 解説: 標準化

kNN やロジスティック回帰では、特徴量のスケールが結果に影響することがあります。そこで、平均を0、標準偏差を1にそろえる標準化を行います。

特徴量 $x$ を標準化した値 $z$ は、次の式で表されます。

$$
z = \frac{x - \mu}{\sigma}
$$

ここで、$\mu$ は平均、$\sigma$ は標準偏差です。単位や値の大きさが違う特徴量を、比較しやすい形にそろえるための処理です。


In [ ]:
# ===== Step 2: 学習データ準備 =====
feature_columns = cancer.feature_names[:10].tolist()  # TODO: 列数を変えて試してよい
X = df[feature_columns]
y = df[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('X_train shape =', X_train.shape)
print('X_test shape =', X_test.shape)


### Step 3: 分類モデルを学習する
ロジスティック回帰、kNN、決定木を学習させ、Accuracy を比べます。Accuracy は全体としてどれくらい当たったかを表します。


#### 解説: Accuracy

Accuracy は、全データのうち正しく分類できた割合です。

$$
Accuracy = \frac{正しく分類した件数}{全件数}
$$

ただし、Accuracy が高くても、重要なクラスを間違えている可能性があります。そのため、次の混同行列で「どのように間違えたか」を確認します。


In [ ]:
# ===== 追加演習: Accuracy を手計算のイメージで確認する =====
example_true = pd.Series([0, 1, 1, 0, 1])
example_pred = pd.Series([0, 1, 0, 0, 1])
manual_accuracy = (example_true == example_pred).mean()
print('正解したか:', (example_true == example_pred).tolist())
print('Accuracy:', manual_accuracy)


In [ ]:
# ===== Step 3: 分類モデル =====
models = {
    'logistic_regression': LogisticRegression(max_iter=1000),
    'kNN': KNeighborsClassifier(n_neighbors=5),
    'decision_tree': DecisionTreeClassifier(random_state=42, max_depth=4),
}

results = []
predictions_by_model = {}

for model_name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    predictions_by_model[model_name] = preds
    results.append({
        'model': model_name,
        'accuracy': accuracy_score(y_test, preds),
    })

results_df = pd.DataFrame(results).sort_values('accuracy', ascending=False)
display(results_df)


In [ ]:
# ===== 追加演習: 特徴量数を変えて比較する =====
# TODO: feature_count を 5, 10, 20, 30 に変えて結果を見る
feature_count = 10
feature_columns_trial = cancer.feature_names[:feature_count].tolist()
X_trial = df[feature_columns_trial]

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_trial, y, test_size=0.2, random_state=42, stratify=y
)
scaler_t = StandardScaler()
X_train_t = scaler_t.fit_transform(X_train_t)
X_test_t = scaler_t.transform(X_test_t)

trial_model = LogisticRegression(max_iter=1000)
trial_model.fit(X_train_t, y_train_t)
trial_preds = trial_model.predict(X_test_t)
print('特徴量数:', feature_count)
print('Accuracy:', accuracy_score(y_test_t, trial_preds))


### Step 4: 混同行列を確認する
予測の間違い方を混同行列で見ます。どのクラスを取り違えやすいか、その間違いにどのような意味があるかを考えてください。


#### 解説: 混同行列

2クラス分類の混同行列では、予測と実際の組み合わせを数えます。

| | 予測: 陰性 | 予測: 陽性 |
| --- | ---: | ---: |
| 実際: 陰性 | TN | FP |
| 実際: 陽性 | FN | TP |

- TP: 陽性を陽性と予測
- TN: 陰性を陰性と予測
- FP: 陰性を陽性と予測
- FN: 陽性を陰性と予測

どの間違いが問題になりやすいかは、データの文脈によって変わります。Accuracy だけでなく、誤分類の種類を見ることが大切です。


In [ ]:
# ===== Step 4: 混同行列 =====
# TODO: model_name を変えて混同行列を比較する
model_name = results_df.iloc[0]['model']
cm = confusion_matrix(y_test, predictions_by_model[model_name])

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=cancer.target_names)
disp.plot(cmap='Blues')
plt.title(f'{model_name} の混同行列')
plt.show()


## 振り返り
- 回帰と分類の違いを、自分の言葉で説明してください。
- どのモデルの Accuracy が高かったですか。
- 混同行列では、どのクラスをどのクラスとして間違えた例がありましたか。
- Accuracy だけで判断しない方がよい理由を説明してください。
